<a href="https://colab.research.google.com/github/ganesh142007/IRS-01.ipynb/blob/main/irs_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from collections import deque

def is_allowed(url, allowed_domains):
    domain = urlparse(url).netloc
    return any(domain.endswith(ad) for ad in allowed_domains)

def crawl_news(topic, allowed_domains, max_pages=20):
    # Seed URLs (you can customize for news portals)
    seeds = [f"https://{domain}" for domain in allowed_domains]

    visited = set()
    queue = deque(seeds)
    results = []

    while queue and len(visited) < max_pages:
        url = queue.popleft()
        if url in visited:
            continue
        try:
            response = requests.get(url, timeout=5)
            if response.status_code != 200:
                continue
            html = response.text
            soup = BeautifulSoup(html, "html.parser")

            # Check if topic is in page text
            if topic.lower() in soup.get_text().lower():
                title = soup.title.string if soup.title else 'No Title'
                results.append((title.strip(), url))
                print(f"Found: {title} - {url}")

            # Extract and enqueue links
            for link_tag in soup.find_all('a', href=True):
                link = urljoin(url, link_tag['href'])
                # Normalize link (remove fragments, query params)
                link_parsed = urlparse(link)
                link = link_parsed.scheme + "://" + link_parsed.netloc + link_parsed.path
                if is_allowed(link, allowed_domains) and link not in visited:
                    queue.append(link)

            visited.add(url)
        except Exception as e:
            # Handle errors like timeouts, connection errors
            continue

    return results

if __name__ == "__main__":
    # User input
    topic = input("Enter the topic to search for: ").strip()
    print("Enter allowed websites (comma-separated, e.g. cnn.com,bbc.com):")
    allowed_input = input().strip()
    allowed_domains = [d.strip() for d in allowed_input.split(",")]

    max_pages = int(input("Enter max number of pages to crawl (e.g. 20): "))

    print(f"\nStarting crawl for topic '{topic}' on domains: {allowed_domains}\n")
    results = crawl_news(topic, allowed_domains, max_pages)

    print("\n--- Crawl Results ---")
    for i, (title, url) in enumerate(results, 1):
        print(f"{i}. {title}\n {url}\n")\

Enter the topic to search for: bbc
Enter allowed websites (comma-separated, e.g. cnn.com,bbc.com):
bbc.com
Enter max number of pages to crawl (e.g. 20): 10

Starting crawl for topic 'bbc' on domains: ['bbc.com']

Found: BBC Home - Breaking News, World News, US News, Sports, Business, Innovation, Climate, Culture, Travel, Video & Audio - https://bbc.com
Found: BBC Home - Breaking News, World News, US News, Sports, Business, Innovation, Climate, Culture, Travel, Video & Audio - https://bbc.com/
Found: BBC News - Breaking news, video and the latest top stories from the U.S. and around the world - https://bbc.com/news
Found: BBC Sport - Scores, Fixtures, News - Live Sport - https://bbc.com/sport
Found: BBC Business | Economy, Tech, AI, Work, Personal Finance, Market news - https://bbc.com/business
Found: BBC Technology | Technology, Health, Environment, AI - https://bbc.com/technology
Found: BBC Health | Nutrition, Exercise, Relationships, Sleep, Longevity - https://bbc.com/health
Found: B